# Réponses de Benoît DAVID & Thomas LECHANOINE

### 1. Choix de l’architecture 

#### Agent apprenant (Q-learning, DQN) :

    Quand le choisir : On choisi le choisir quand on ne connaît pas les règles exactes, l'environnement est trop complexe pour coder des règles "en dur", ou si l'on veut que l'IA découvre des stratégies inédites.

    Argument : "Idéal pour généraliser et s'adapter à des situations non prévues par le programmeur. L'agent apprendra de ses erreurs et s'adaptera sa politique pour maximiser ses récompenses futures."

####  Description des composants


    1. Le Module de Perception (L'État s)

        C'est les "yeux" de l'IA. Il simplifie le jeu en une liste de nombres (ex: coordonnées X,Y, distance à l'ennemi). On appelle cela l'État courant.

    2. La Fonction de Valeur (Le "Cerveau" Q)

        C'est le cœur du système. Il doit répondre à la question : "Dans cet état, combien de points vais-je gagner si je fais cette action ?".

        En Q-Learning simple : C'est une Q-Table (un grand tableau Excel) qui stocke les valeurs.

        En DQN : C'est un Réseau de Neurones qui calcule ces valeurs.

    3. La Politique de Décision (ϵ-greedy)

        C'est le module qui choisit l'action finale. Il a deux modes :

            Exploitation : Choisir la meilleure action connue (celle avec la plus grande valeur Q).

            Exploration : Choisir une action au hasard (pour découvrir de nouvelles stratégies).

    4. Le Module d'Apprentissage (Mise à jour)

        C'est le mécanisme qui corrige l'IA. Après avoir reçu une Récompense du jeu, ce module met à jour la Q-Table ou le Réseau pour que l'IA ne refasse pas les mêmes erreurs (ou répète les bons coups).

#### Comment cette architecture s'interface avec le jeu ?

Pour un agent apprenant, l'interface est une boucle en 4 étapes (c'est le standard de l'Apprentissage par Renforcement) :

    Input : Le jeu envoie l'État actuel (S) à l'agent.

    Action : L'agent consulte sa Q-Table/Réseau et envoie une Action (A) au jeu.

    Feedback : Le jeu exécute l'action et renvoie deux choses à l'agent :

        Le Nouvel État (S′) (la situation a changé).

        Une Récompense (R) (points gagnés, pénalité si mort, ou 0).

    Learning : L'agent utilise (S,A,R,S′) pour apprendre, puis recommence la boucle.

### 2. Problématique d’apprentissage Problématique choisie : Généralisation et Robustesse face à la taille de la grille.

#### Formulation : 
    "Comment entraîner un agent sur une petite grille (ex: 10x10) pour qu'il soit performant sur une grille beaucoup plus grande (ex: 50x50) sans réentraînement ?"

#### Pourquoi est-ce intéressant/challenging ?

       Overfitting (Surapprentissage) : Un agent classique a tendance à mémoriser des chemins spécifiques ("au pixel 4, tourne à droite"). Si on change la taille de la grille, ses repères absolus disparaissent.

        Abstraction : 
        Pour réussir, l'agent ne doit pas apprendre "où il est", mais "ce qui l'entoure". Il doit apprendre des concepts locaux (ex: "il y a un mur à 2 cases devant") plutôt que globaux.

        Économie de calcul : 
        Entraîner sur une grande grille est très coûteux (l'espace d'états explose). Si l'on peut entraîner sur du petit pour jouer sur du grand, on gagne énormément en temps de calcul.

### 3. Intégration avec l’algorithme génétique

    C'est ici que l'on crée un système hybride puissant. L'algorithme génétique (AG) ne va pas remplacer l'apprentissage par renforcement, il va le superviser.

#### Comment combiner les deux ? (Approche Neuro-Évolutionnaire Hybride) 
    Je propose d'utiliser l'Algorithme Génétique pour optimiser les Hyperparamètres et la Structure du réseau, tandis que le DQN (Backpropagation) optimise les Poids du réseau.

#### Aspects optimisés génétiquement (Le "Génome" de l'agent) : 
    Chaque individu de la population génétique est défini par un vecteur de gènes contenant :

    Taux d'apprentissage (Learning Rate) : À quelle vitesse le cerveau modifie ses connexions.

    Facteur d'actualisation (Gamma) : L'importance donnée au futur vs l'immédiat.

    Architecture du réseau : Nombre de couches cachées, nombre de neurones par couche.

    Fonction de récompense (Reward Shaping) : Exemple : poids de la pénalité "tourner en rond" vs poids du bonus "se rapprocher de la nourriture".

#### Schéma d'interaction :Population Initiale (AG) : 
    1. Création de 20 agents avec des hyperparamètres aléatoires.
    2. Évaluation (RL) : Chaque agent s'entraîne avec sa propre architecture/paramètres pendant $N$ épisodes (ex: 1000 parties).Note : C'est l'étape coûteuse.

    3. Calcul du Fitness : Le score moyen obtenu par l'agent lors des 100 dernières parties définit son "Fitness".
    4. Sélection & Reproduction (AG) : On garde les agents avec les meilleurs scores. 
    On croise leurs hyperparamètres (ex: prendre le Learning Rate du père et l'Architecture de la mère).

    5. Mutation : On modifie légèrement certains paramètres au hasard pour explorer de nouvelles pistes.
    6. Boucle : On recommence à l'étape 2 avec la nouvelle génération.
    
#### Gestion du compromis Temps/Performance : L'entraînement complet de chaque individu par RL est trop long. Voici comment optimiser/      
    . Parallélisme : Lancer les entraînements des 20 agents en parallèle sur différents cœurs CPU/GPU.
    . Entraînement partiel (Early Stopping) : Ne pas entraîner l'agent jusqu'à la perfection pendant la phase génétique. On l'entraîne juste assez pour voir sa courbe de progression. Si un agent apprend vite, il a un bon génome.
    . Héritage des poids (Lamarckisme) : Un enfant peut hériter non seulement des hyperparamètres de ses parents, mais aussi d'une partie de leurs "poids" neuronaux déjà appris, pour ne pas repartir de zéro (Transfer Learning).
